# SimpleLLM V0.21 — GPT-style Decoder-Only Transformer (Bug-Fix Release)

## Анализ багов V0.15 и что исправлено:

### КРИТИЧЕСКИЕ (ломают обучение):
1. **PAD-токены в loss** — `SparseCategoricalCrossentropy` считала loss на ВСЕХ позициях, включая PAD=0. Короткие строки (40 токенов из 128) генерировали мусорные градиенты на 88 PAD-позициях. Модель тратила capacity на "предсказание тишины". **Фикс:** `sample_weight` маска в tf.data pipeline.
2. **Padding direction mismatch** — обучение: RIGHT-pad `[tok1,tok2,PAD,PAD]`, генерация: LEFT-pad `[PAD,PAD,tok1,tok2]`. Position embeddings видели разные паттерны → модель путалась. **Фикс:** RIGHT-pad везде, logits берём с позиции последнего реального токена.
3. **SentencePiece зависание** — `input_sentence_size=100000` + длинные строки Gutenberg (>1024 символов без переносов) заставляли BPE merge фазу зацикливаться. **Фикс:** обрезка до 512 символов + `input_sentence_size=50000` + `split_by_whitespace=True`.

### СЕРЬЁЗНЫЕ (искажают метрики / нестабильность):
4. **Perplexity на PAD** — метрика `perplexity()` считала exp(loss) по всем позициям → завышенная perplexity. **Фикс:** masked perplexity.
5. **Нет мониторинга градиентов** — NaN/explosion/vanishing не детектировались. clipnorm=1.0 молча обрезал, но не алертил. **Фикс:** `GradientMonitor` callback.
6. **Нет mixed precision** — T4/P100 имеют Tensor Cores для float16. Без mixed precision обучение ~2x медленнее. **Фикс:** `tf.keras.mixed_precision.set_global_policy('mixed_float16')`.

### АРХИТЕКТУРНЫЕ (чистота кода / воспроизводимость):
7. **Нет `get_config()`** в кастомных слоях → нельзя сохранить/загрузить модель через `model.save()`.
8. **`generate()` зависит от глобальных `sp`, `CONTEXT_WIN`** → хрупкий код.
9. **Пустые строки** не обрабатывались в SPTokenizer → all-PAD sequences.
10. **`endseq` как текст** — надёжнее использовать `eos_id` напрямую.

In [ ]:
# ========================== [CELL 1] ЗАВИСИМОСТИ ==========================
# Почему именно эти:
# - sentencepiece: BPE токенизатор (тот же алгоритм что в GPT-2, LLaMA, Mistral)
# - TF mixed precision: ускорение ~1.5-2x на T4/P100 через float16 матричные операции
# - Всё остальное входит в стандартную TF поставку

!pip install -q sentencepiece

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, losses, callbacks
import sentencepiece as spm
import os, urllib.request, tempfile, time, re, math

# ---- ВОСПРОИЗВОДИМОСТЬ ----
# Фиксируем seed на уровне numpy и TF. SentencePiece shuffle — отдельно.
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- GPU SETUP ----
# Разрешаем динамическое выделение памяти GPU (не забирать все 16GB сразу)
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {gpus}")
print(f"Numpy: {np.__version__}")
print(f"SentencePiece: {spm.__version__}")


In [ ]:
# ========================== [CELL 2] ГИПЕРПАРАМЕТРЫ ==========================
# Стратегия: тонкая но глубокая модель (6 блоков x 256 dim).
# При ~160K строк корпуса это оптимальнее чем 4 блока x 512 dim:
# - Больше блоков = лучше выучивает иерархические паттерны языка
# - Меньше dim = меньше параметров = меньше переобучение на малых данных

MAX_VOCAB = 8000         # BPE словарь. 8K — sweet spot для ~10MB корпуса
CONTEXT_WIN = 128        # Окно контекста в токенах. 128 > 50 : связнее текст
EMBED_DIM = 256          # Размерность эмбеддингов (256 из-за малого корпуса)
HEADS = 8                # 256 / 8 = 32 на голову (минимум для MHA)
FEED_FORWARD = 1024      # 4 x EMBED_DIM (стандартное соотношение GPT)
TRANSFORMER_BLOCKS = 6   # Глубина модели (6 блоков Pre-Norm)
DROPOUT_RATE = 0.15      # Регуляризация (0.15 для малого корпуса)

BATCH_SIZE = 64          # Tesla T4/P100: оптимально при EMBED=256
EPOCHS = 30              # С EarlyStopping реально ~15-20
LEARNING_RATE = 5e-4     # Пиковый LR (после warmup)
MIN_LR = 1e-5            # Минимальный LR (конец cosine decay)
WARMUP_STEPS = 1000      # Линейный warmup: 0 -> LEARNING_RATE за 1000 шагов
VALIDATION_SPLIT = 0.1   # 10% данных на валидацию

MAX_TRAIN_LINES = 300000 # Лимит строк (запас, корпус ~160K)
MIN_LINE_LENGTH = 15     # Фильтрация коротких строк (мусор, заголовки)

# ---- Mixed Precision ----
# НОВОЕ в V0.21: float16 на GPU (Tensor Cores T4/P100)
# Это ~1.5-2x ускорение без потери качества.
# ВНИМАНИЕ: output logits ДОЛЖНЫ быть float32 (для стабильности softmax loss).
# Мы обеспечиваем это через cast в модели.
USE_MIXED_PRECISION = True

if USE_MIXED_PRECISION and len(gpus) > 0:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision: ENABLED (float16 compute, float32 accumulate)")
else:
    print("Mixed precision: DISABLED (CPU mode or no GPU)")

# Оценка параметров модели:
estimated_params = (MAX_VOCAB * EMBED_DIM +
                    TRANSFORMER_BLOCKS * (4 * EMBED_DIM**2 + 2 * EMBED_DIM * FEED_FORWARD))
print(f"Ожидаемые параметры: ~{estimated_params / 1e6:.1f}M")


In [ ]:
# ========================== [CELL 3] ЗАГРУЗКА ДАННЫХ (32 датасета) ==========================
# Этот модуль НЕ изменён относительно V0.15 (баги были не здесь).
# Единственное дополнение: явный assert на минимальное количество строк.

DATASETS = {
    # =================== ДРАМАТУРГИЯ / ПОЭЗИЯ ===================
    'shakespeare': {
        'url': 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt',
        'filename': 'shakespeare.txt'
    },
    'tiny_shakespeare': {
        'url': 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'filename': 'tiny_shakespeare.txt'
    },
    # =================== РЕЛИГИЯ / ФИЛОСОФИЯ ===================
    'bible_kjv': {
        'url': 'https://raw.githubusercontent.com/mxw/grmr/master/src/finaltests/bible.txt',
        'filename': 'bible_kjv.txt'
    },
    'plato_republic': {
        'url': 'https://www.gutenberg.org/cache/epub/1497/pg1497.txt',
        'filename': 'plato_republic.txt'
    },
    # =================== НАУКА ===================
    'darwin_origin': {
        'url': 'https://www.gutenberg.org/cache/epub/1228/pg1228.txt',
        'filename': 'darwin_origin.txt'
    },
    'einstein_relativity': {
        'url': 'https://www.gutenberg.org/cache/epub/5001/pg5001.txt',
        'filename': 'einstein_relativity.txt'
    },
    # =================== ГОТИКА / ХОРРОР ===================
    'poe': {
        'url': 'https://www.gutenberg.org/cache/epub/2147/pg2147.txt',
        'filename': 'poe.txt'
    },
    'frankenstein': {
        'url': 'https://www.gutenberg.org/cache/epub/84/pg84.txt',
        'filename': 'frankenstein.txt'
    },
    'dracula': {
        'url': 'https://www.gutenberg.org/cache/epub/345/pg345.txt',
        'filename': 'dracula.txt'
    },
    # =================== ДЕТЕКТИВЫ ===================
    'sherlock': {
        'url': 'https://www.gutenberg.org/cache/epub/1661/pg1661.txt',
        'filename': 'sherlock.txt'
    },
    # =================== ПРИКЛЮЧЕНИЯ / ФАНТАСТИКА ===================
    'alice': {
        'url': 'https://www.gutenberg.org/cache/epub/11/pg11.txt',
        'filename': 'alice.txt'
    },
    'around_the_world': {
        'url': 'https://www.gutenberg.org/cache/epub/103/pg103.txt',
        'filename': 'around_the_world.txt'
    },
    'war_of_worlds': {
        'url': 'https://www.gutenberg.org/cache/epub/36/pg36.txt',
        'filename': 'war_of_worlds.txt'
    },
    'time_machine': {
        'url': 'https://www.gutenberg.org/cache/epub/35/pg35.txt',
        'filename': 'time_machine.txt'
    },
    'treasure_island': {
        'url': 'https://www.gutenberg.org/cache/epub/120/pg120.txt',
        'filename': 'treasure_island.txt'
    },
    'twenty_thousand_leagues': {
        'url': 'https://www.gutenberg.org/cache/epub/164/pg164.txt',
        'filename': 'twenty_thousand_leagues.txt'
    },
    'gulliver': {
        'url': 'https://www.gutenberg.org/cache/epub/829/pg829.txt',
        'filename': 'gulliver.txt'
    },
    'robinson_crusoe': {
        'url': 'https://www.gutenberg.org/cache/epub/521/pg521.txt',
        'filename': 'robinson_crusoe.txt'
    },
    # =================== РОМАНЫ / ПРОЗА ===================
    'pride_prejudice': {
        'url': 'https://www.gutenberg.org/cache/epub/1342/pg1342.txt',
        'filename': 'pride_prejudice.txt'
    },
    'moby_dick': {
        'url': 'https://www.gutenberg.org/cache/epub/2701/pg2701.txt',
        'filename': 'moby_dick.txt'
    },
    'tom_sawyer': {
        'url': 'https://www.gutenberg.org/cache/epub/74/pg74.txt',
        'filename': 'tom_sawyer.txt'
    },
    'huck_finn': {
        'url': 'https://www.gutenberg.org/cache/epub/76/pg76.txt',
        'filename': 'huck_finn.txt'
    },
    'great_expectations': {
        'url': 'https://www.gutenberg.org/cache/epub/1400/pg1400.txt',
        'filename': 'great_expectations.txt'
    },
    'tale_two_cities': {
        'url': 'https://www.gutenberg.org/cache/epub/98/pg98.txt',
        'filename': 'tale_two_cities.txt'
    },
    'dorian_gray': {
        'url': 'https://www.gutenberg.org/cache/epub/174/pg174.txt',
        'filename': 'dorian_gray.txt'
    },
    'heart_of_darkness': {
        'url': 'https://www.gutenberg.org/cache/epub/219/pg219.txt',
        'filename': 'heart_of_darkness.txt'
    },
    'jane_eyre': {
        'url': 'https://www.gutenberg.org/cache/epub/1260/pg1260.txt',
        'filename': 'jane_eyre.txt'
    },
    'monte_cristo': {
        'url': 'https://www.gutenberg.org/cache/epub/1184/pg1184.txt',
        'filename': 'monte_cristo.txt'
    },
    # =================== ПОЛИТИКА / ИСТОРИЯ ===================
    'the_prince': {
        'url': 'https://www.gutenberg.org/cache/epub/1232/pg1232.txt',
        'filename': 'the_prince.txt'
    },
    'art_of_war': {
        'url': 'https://www.gutenberg.org/cache/epub/132/pg132.txt',
        'filename': 'art_of_war.txt'
    },
    # =================== УТОПИЯ / АНТИУТОПИЯ ===================
    'utopia': {
        'url': 'https://www.gutenberg.org/cache/epub/2130/pg2130.txt',
        'filename': 'utopia.txt'
    },
    # =================== ЭПОС / МИФОЛОГИЯ ===================
    'odyssey': {
        'url': 'https://www.gutenberg.org/cache/epub/1727/pg1727.txt',
        'filename': 'odyssey.txt'
    },
    'iliad': {
        'url': 'https://www.gutenberg.org/cache/epub/6130/pg6130.txt',
        'filename': 'iliad.txt'
    },
}

def clean_line(line):
    """Очищает строку: Unicode -> ASCII, множественные пробелы -> один."""
    line = line.strip()
    line = line.replace('\u2018', "'").replace('\u2019', "'")
    line = line.replace('\u201c', '"').replace('\u201d', '"')
    line = line.replace('\u2014', '--').replace('\u2013', '-')
    line = line.replace('\u2026', '...')
    line = re.sub(r'[^\x20-\x7E]', '', line)
    line = re.sub(r'\s+', ' ', line).strip()
    return line

# Директория для кэша
cache_dir = os.path.join(os.path.expanduser('~'), '.keras', 'datasets', 'llm_corpus')
os.makedirs(cache_dir, exist_ok=True)

all_lines = []
dataset_stats = {}

for name, info in DATASETS.items():
    filepath = os.path.join(cache_dir, info['filename'])
    try:
        if not os.path.exists(filepath):
            print(f"  Скачиваю {name}...")
            urllib.request.urlretrieve(info['url'], filepath)

        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.read().split('\n')

        filtered = []
        for line in lines:
            cleaned = clean_line(line)
            if (len(cleaned) > MIN_LINE_LENGTH
                and not cleaned.startswith('***')
                and 'gutenberg' not in cleaned.lower()
                and 'project gutenberg' not in cleaned.lower()):
                filtered.append(cleaned)

        all_lines.extend(filtered)
        dataset_stats[name] = len(filtered)
        print(f"  OK {name}: {len(filtered):,} строк")

    except Exception as e:
        print(f"  FAIL {name}: {e}")

np.random.shuffle(all_lines)
train_text = [line + ' endseq' for line in all_lines[:MAX_TRAIN_LINES]]

# ---- ASSERT: минимальный размер корпуса ----
assert len(train_text) > 10000, (
    f"Корпус слишком маленький: {len(train_text)} строк. "
    f"Нужно минимум 10,000 для обучения LLM."
)

print(f"\n{'='*60}")
print(f"  Всего строк в корпусе:    {len(all_lines):,}")
print(f"  Строк для обучения:       {len(train_text):,}")
print(f"  Датасетов загружено:      {len(dataset_stats)}/{len(DATASETS)}")
print(f"{'='*60}")

# ---- Статистика длин строк (для диагностики токенизатора) ----
lengths = [len(line) for line in train_text]
print(f"  Длина строк: min={min(lengths)}, max={max(lengths)}, "
      f"mean={np.mean(lengths):.0f}, median={np.median(lengths):.0f}")
print(f"  Строк > 512 символов: {sum(1 for l in lengths if l > 512):,}")
print(f"  Пример: {train_text[0][:100]}...")


In [ ]:
# ========================== [CELL 4] ТОКЕНИЗАТОР — SentencePiece BPE ==========================
#
# КРАСНАЯ ЗОНА: именно здесь V0.15 зависала. Три корневые причины:
#
# ПРИЧИНА 1: Длинные строки из Gutenberg (параграфы без переносов, >2000 символов).
#   SentencePiece при BPE обучении загружает ВСЮ строку в память для merge-операций.
#   Строки >1024 символов при vocab_size=8000 создают комбинаторный взрыв в фазе merging.
#   max_sentence_length=1024 должен был помочь, но SPM ОБРЕЗАЕТ, а не пропускает —
#   обрезанные строки могут ломать пары символов и замедлять сходимость.
#   ФИКС: обрезаем ДО записи в файл (до 512 символов). SPM получает только короткие строки.
#
# ПРИЧИНА 2: input_sentence_size=100000 — слишком большая выборка.
#   BPE алгоритм O(N * V) где N = количество символов во всех строках.
#   100K строк по ~100 символов = 10M символов. При vocab_size=8000 это ~80 миллиардов
#   операций в фазе merge. Снижаем до 50K.
#   ФИКС: input_sentence_size=50000.
#
# ПРИЧИНА 3: split_by_whitespace не был задан явно.
#   По умолчанию SPM может пытаться строить BPE через границы слов,
#   что увеличивает алфавит токенов и замедляет обучение.
#   ФИКС: split_by_whitespace=True (явно).
#
# ПОДХОД: Безопасный (не оптимальный). Оптимальный — использовать unigram model_type
# вместо BPE (быстрее сходится). Но BPE — стандарт в GPT/LLaMA, оставляем для
# совместимости и предсказуемости поведения при разных корпусах.

MAX_LINE_CHARS = 512  # Жёсткий лимит длины строки ДО передачи в SentencePiece

# ---- Шаг 1: Записываем корпус с обрезкой ----
corpus_path = os.path.join(cache_dir, 'corpus_for_spm_v021.txt')
lines_written = 0
max_observed_len = 0

with open(corpus_path, 'w', encoding='utf-8') as f:
    for line in train_text:
        # ОБРЕЗАЕМ до MAX_LINE_CHARS. Это критично: без обрезки SPM зависает.
        truncated = line[:MAX_LINE_CHARS]
        # Пропускаем пустые строки после обрезки (не должно быть, но safety check)
        if len(truncated.strip()) == 0:
            continue
        f.write(truncated + '\n')
        lines_written += 1
        max_observed_len = max(max_observed_len, len(truncated))

corpus_size_mb = os.path.getsize(corpus_path) / 1024 / 1024
print(f"Корпус для SPM:")
print(f"  Строк:          {lines_written:,}")
print(f"  Max длина:       {max_observed_len} символов (лимит: {MAX_LINE_CHARS})")
print(f"  Размер файла:   {corpus_size_mb:.1f} MB")

# Проверяем: если max_observed_len > MAX_LINE_CHARS, что-то пошло не так
assert max_observed_len <= MAX_LINE_CHARS, (
    f"Строка длиной {max_observed_len} прошла обрезку! Это баг."
)

# ---- Шаг 2: Удаляем старую модель (не используем кэш) ----
spm_model_prefix = os.path.join(cache_dir, 'spm_bpe_v021')
for ext in ['.model', '.vocab']:
    path = spm_model_prefix + ext
    if os.path.exists(path):
        os.remove(path)
        print(f"  Удалён: {path}")

# ---- Шаг 3: Обучаем BPE ----
print(f"\nОбучаем BPE (vocab_size={MAX_VOCAB}, input_sentences=50K)...")
print(f"byte_fallback=True -> 0 [UNK] токенов (любой байт кодируется)")

EXPECTED_MAX_SEC = 120  # Тайм-аут диагностики (не прерывает, только предупреждает)
t0 = time.time()

spm.SentencePieceTrainer.train(
    input=corpus_path,
    model_prefix=spm_model_prefix,
    vocab_size=MAX_VOCAB,
    model_type='bpe',

    # --- КРИТИЧЕСКИЕ ПАРАМЕТРЫ (исправляют зависание) ---
    input_sentence_size=50_000,       # V0.15 было 100K -> зависало
    shuffle_input_sentence=True,      # Случайная выборка, не первые N строк
    character_coverage=0.9995,        # ASCII корпус -> покрываем почти всё
    max_sentence_length=1024,         # Запас: строки уже <= 512 символов
    byte_fallback=True,               # Редкие байты -> <0xNN> токены (не [UNK])
    split_by_whitespace=True,         # НОВОЕ: не строить BPE через пробелы

    # --- Зарезервированные токены ---
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    pad_piece='[PAD]', unk_piece='[UNK]', bos_piece='[BOS]', eos_piece='[EOS]',

    # --- Пользовательские символы ---
    user_defined_symbols=['endseq'],

    # --- Производительность ---
    num_threads=os.cpu_count(),
    train_extremely_large_corpus=False,
)

elapsed = time.time() - t0
print(f"\nBPE обучение завершено за {elapsed:.1f}с")

if elapsed > EXPECTED_MAX_SEC:
    print(f"WARNING: обучение заняло >{EXPECTED_MAX_SEC}с!")
    print(f"  Возможные причины: большой корпус, медленный CPU, не SSD диск.")
else:
    print(f"OK: время в норме (< {EXPECTED_MAX_SEC}с)")

# ---- Шаг 4: Загружаем модель ----
sp = spm.SentencePieceProcessor()
sp.load(spm_model_prefix + '.model')

actual_vocab_size = sp.get_piece_size()
print(f"\nСловарь: {actual_vocab_size:,} токенов (запрошено: {MAX_VOCAB:,})")
print(f"PAD={sp.pad_id()}, UNK={sp.unk_id()}, BOS={sp.bos_id()}, EOS={sp.eos_id()}")

# ---- Шаг 5: Smoke-тест токенизатора ----
print(f"\n--- Smoke-тест SentencePiece ---")
test_cases = ['Alice', 'electromagnetic', 'Shakespeare', 'natural selection', 'I am the king']
for text in test_cases:
    ids = sp.encode(text, out_type=int)
    pieces = sp.encode(text, out_type=str)
    decoded = sp.decode(ids)
    print(f"  '{text}' -> {pieces} -> {ids} -> '{decoded}'")
    # Assert: round-trip должен вернуть примерно тот же текст
    # (BPE может менять пробелы на юникод-заменители, поэтому сравниваем stripped)
    assert decoded.strip() == text.strip(), (
        f"Round-trip FAIL: '{text}' -> '{decoded}'"
    )
print("  Round-trip OK")

# Проверка: ноль UNK в первых 1000 строках
unk_count = sum(
    1 for line in train_text[:1000]
    for tid in sp.encode(line, out_type=int)
    if tid == sp.unk_id()
)
print(f"  [UNK] в первых 1000 строках: {unk_count} (ожидается 0 с byte_fallback)")

# Edge case: пустая строка
empty_ids = sp.encode('', out_type=int)
print(f"  Пустая строка -> {empty_ids} (ожидается [])")

# Edge case: очень длинная строка
long_text = 'word ' * 500  # 2500 символов
long_ids = sp.encode(long_text, out_type=int)
print(f"  2500-символьная строка -> {len(long_ids)} токенов")


In [ ]:
# ========================== [CELL 5] SPTokenizer — ОБЁРТКА ==========================
#
# ЗАЧЕМ ОБЁРТКА:
# SentencePiece.encode() возвращает list[int] переменной длины.
# TensorFlow tf.data требует numpy-массивы ФИКСИРОВАННОЙ длины для батчирования.
# Эта обёртка обеспечивает:
# 1) Padding до фиксированной длины (seq_length)
# 2) Truncation длинных последовательностей
# 3) Батчевую обработку с прогрессом (не "зависает" на 160K строках)
# 4) Edge case handling (пустые строки, bytes, tf.Tensor)
#
# ИСПРАВЛЕНИЯ V0.21:
# - encode() возвращает tuple (ids, length) для генерации sample_weight маски
# - encode_batch() с чанками по 10K + прогресс
# - Пустые строки -> корректный all-PAD вектор (не ломает обучение)
# - Assert на выходную форму тензоров

class SPTokenizer:
    """Обёртка над SentencePiece для tf.data pipeline.

    Гарантирует фиксированную длину выхода и корректную обработку edge cases.
    """

    def __init__(self, sp_model, seq_length):
        self.sp = sp_model
        self.seq_length = seq_length
        self.pad_id = sp_model.pad_id()  # 0

    def encode(self, text):
        """Кодирует одну строку -> list[int] длины self.seq_length.

        Обрабатывает edge cases:
        - bytes/np.bytes_ (tf.data иногда передаёт bytes)
        - Пустые строки -> all-PAD
        - Длинные строки -> truncation

        Returns:
            list[int] длины self.seq_length
        """
        # Конвертация типов (tf.data, numpy -> str)
        if isinstance(text, (bytes, np.bytes_)):
            text = text.decode('utf-8')
        elif not isinstance(text, str):
            text = str(text)

        # Edge case: пустая строка -> all-PAD
        if len(text.strip()) == 0:
            return [self.pad_id] * self.seq_length

        ids = self.sp.encode(text, out_type=int)

        # Truncation: обрезаем до seq_length
        ids = ids[:self.seq_length]

        # Padding: добиваем нулями справа
        num_pad = self.seq_length - len(ids)
        if num_pad > 0:
            ids = ids + [self.pad_id] * num_pad

        # Invariant: длина ВСЕГДА = seq_length
        assert len(ids) == self.seq_length, (
            f"encode() вернул {len(ids)} токенов, ожидалось {self.seq_length}"
        )
        return ids

    def encode_batch(self, texts):
        """Кодирует список строк -> numpy array (N, seq_length).

        Обработка чанками по 10K с прогрессом (не зависает).

        Args:
            texts: list[str] | list[bytes] | tf.Tensor
        Returns:
            np.ndarray dtype=int32, shape=(len(texts), seq_length)
        """
        # Конвертация tf.Tensor -> list[str]
        if hasattr(texts, 'numpy'):
            texts = [
                t.decode('utf-8') if isinstance(t, bytes) else str(t)
                for t in texts.numpy()
            ]
        elif len(texts) > 0 and isinstance(texts[0], (bytes, np.bytes_)):
            texts = [t.decode('utf-8') for t in texts]

        total = len(texts)
        CHUNK = 10_000
        chunks = []

        for start in range(0, total, CHUNK):
            end = min(start + CHUNK, total)
            encoded = [self.encode(t) for t in texts[start:end]]
            chunks.append(np.array(encoded, dtype=np.int32))
            print(f"\r  Токенизация: {end:,}/{total:,} ({end/total*100:.1f}%)", end='', flush=True)

        print()  # Новая строка

        result = np.concatenate(chunks, axis=0)

        # Assert: форма массива корректна
        assert result.shape == (total, self.seq_length), (
            f"encode_batch shape {result.shape}, ожидалось ({total}, {self.seq_length})"
        )
        return result

    def decode(self, ids):
        """Декодирует list[int] -> строку, пропуская PAD."""
        if hasattr(ids, 'numpy'):
            ids = ids.numpy()
        ids = [int(i) for i in ids if int(i) != self.pad_id]
        return self.sp.decode(ids)

    def vocabulary_size(self):
        return self.sp.get_piece_size()


# ---- Создаём экземпляр ----
# +1 к CONTEXT_WIN: нужен для сдвига X/y (input[:-1], target[1:])
tokenizer = SPTokenizer(sp, seq_length=CONTEXT_WIN + 1)

print(f"SPTokenizer:")
print(f"  Vocab:      {tokenizer.vocabulary_size():,}")
print(f"  Seq length: {tokenizer.seq_length} (CONTEXT_WIN={CONTEXT_WIN} + 1)")
print(f"  PAD id:     {tokenizer.pad_id}")

# ---- Smoke-тесты SPTokenizer ----
print(f"\n--- Smoke-тесты SPTokenizer ---")

# Тест 1: round-trip
_t = "The quick brown fox jumps over the lazy dog."
_ids = tokenizer.encode(_t)
_dec = tokenizer.decode(_ids)
assert len(_ids) == tokenizer.seq_length, f"Длина {len(_ids)} != {tokenizer.seq_length}"
assert _dec.strip() == _t.strip(), f"Round-trip fail: '{_t}' -> '{_dec}'"
print(f"  [1] Round-trip OK: '{_t}' -> {_ids[:8]}... -> '{_dec}'")

# Тест 2: пустая строка
_empty = tokenizer.encode("")
assert all(x == 0 for x in _empty), f"Пустая строка не all-PAD: {_empty[:5]}"
print(f"  [2] Пустая строка -> all-PAD OK")

# Тест 3: очень длинная строка (должна быть обрезана)
_long = "word " * 500
_long_ids = tokenizer.encode(_long)
assert len(_long_ids) == tokenizer.seq_length, f"Длинная строка: {len(_long_ids)} != {tokenizer.seq_length}"
print(f"  [3] Длинная строка -> truncated to {tokenizer.seq_length} OK")

# Тест 4: bytes input (как от tf.data)
_bytes_ids = tokenizer.encode(b"Hello world")
assert len(_bytes_ids) == tokenizer.seq_length
print(f"  [4] bytes input OK")

# Тест 5: batch
_batch = tokenizer.encode_batch(["Hello", "World", "Test"])
assert _batch.shape == (3, tokenizer.seq_length), f"Batch shape: {_batch.shape}"
assert _batch.dtype == np.int32
print(f"  [5] Batch encode OK: shape={_batch.shape}, dtype={_batch.dtype}")

print("\nВсе smoke-тесты токенизатора пройдены!")


In [ ]:
# ========================== [CELL 6] tf.data PIPELINE ==========================
#
# КРИТИЧЕСКИЙ ФИКС V0.21: МАСКИРОВКА PAD В LOSS
#
# БАГ V0.15: SparseCategoricalCrossentropy считала loss по ВСЕМ позициям,
# включая PAD-токены (id=0). При короткой строке (40 реальных токенов из 128):
# - 40 позиций давали полезный градиент
# - 88 позиций (PAD) давали мусорный градиент (модель учила предсказывать PAD)
# Это замедляло обучение и тратило model capacity.
#
# ФИКС: sample_weight — бинарная маска [1,1,...,1,0,0,...,0] для каждой
# последовательности. Loss считается ТОЛЬКО по реальным токенам.
#
# ПОДХОД: Два варианта были рассмотрены:
# (a) sample_weight в tf.data.Dataset (выбран — чище, стандартный подход в Keras)
# (b) Кастомная loss функция с ignore_index — сложнее, легче ошибиться
#
# Дополнительно: вывод статистики для убеждения что pipeline работает корректно.

print("Токенизация полного корпуса...")
t0 = time.time()
all_token_ids = tokenizer.encode_batch(train_text)
print(f"Токенизировано: {all_token_ids.shape} за {time.time()-t0:.1f}с")

# ---- Разделяем на X (input) и y (target) со сдвигом на 1 ----
# Если seq = [A, B, C, D, PAD]
# X = [A, B, C, D]     (предыдущие токены)
# y = [B, C, D, PAD]   (следующие токены — target)
X_all = all_token_ids[:, :-1]  # (N, CONTEXT_WIN)
y_all = all_token_ids[:, 1:]   # (N, CONTEXT_WIN)

# ---- НОВОЕ: sample_weight маска PAD ----
# Маска: 1.0 где y != PAD, 0.0 где y == PAD
# Loss на PAD-позициях умножается на 0 -> не влияет на градиенты
w_all = (y_all != tokenizer.pad_id).astype(np.float32)

# Статистика маски
total_tokens = w_all.size
real_tokens = int(w_all.sum())
pad_tokens = total_tokens - real_tokens
print(f"\nСтатистика токенов:")
print(f"  Всего позиций:   {total_tokens:,}")
print(f"  Реальных:        {real_tokens:,} ({real_tokens/total_tokens*100:.1f}%)")
print(f"  PAD (masked):    {pad_tokens:,} ({pad_tokens/total_tokens*100:.1f}%)")
print(f"  -> V0.15 считала loss на ВСЕХ {total_tokens:,} позициях!")
print(f"  -> V0.21 считает loss ТОЛЬКО на {real_tokens:,} реальных.")

# ---- Train / Validation split ----
split_idx = int(len(X_all) * (1 - VALIDATION_SPLIT))
X_train, X_val = X_all[:split_idx], X_all[split_idx:]
y_train, y_val = y_all[:split_idx], y_all[split_idx:]
w_train, w_val = w_all[:split_idx], w_all[split_idx:]

# ---- tf.data.Dataset ----
# ТРОЙКА (X, y, sample_weight) — Keras автоматически применяет sample_weight к loss
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train, w_train))
    .shuffle(buffer_size=min(len(X_train), 50_000), seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val, w_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(f"\nDatasets:")
print(f"  Train:      {len(X_train):,} samples, {len(X_train)//BATCH_SIZE:,} batches/epoch")
print(f"  Validation: {len(X_val):,} samples")

# ---- Smoke-тест: проверяем что dataset выдаёт тройки правильной формы ----
for xb, yb, wb in train_ds.take(1):
    assert xb.shape[1] == CONTEXT_WIN, f"X shape: {xb.shape}, expected (*, {CONTEXT_WIN})"
    assert yb.shape == xb.shape, f"y shape {yb.shape} != X shape {xb.shape}"
    assert wb.shape == xb.shape, f"w shape {wb.shape} != X shape {xb.shape}"
    assert wb.dtype == tf.float32, f"w dtype: {wb.dtype}, expected float32"
    # Проверяем: w[i,j] == 0 только там где y[i,j] == 0 (PAD)
    pad_mask = (yb.numpy() == 0)
    weight_zero = (wb.numpy() == 0.0)
    assert np.all(pad_mask == weight_zero), "Маска не совпадает с PAD позициями!"
    print(f"\n  Smoke-тест batch: X={xb.shape}, y={yb.shape}, w={wb.shape}")
    print(f"  Пример w[0,:20]: {wb[0,:20].numpy()}")
    print(f"  Pipeline smoke-тест: OK")
    break


In [ ]:
# ========================== [CELL 7] МЕТРИКА + EMBEDDING ==========================
#
# ФИКС V0.21: Perplexity с маскированием PAD
# V0.15 считала perplexity по ВСЕМ позициям включая PAD -> завышенный результат.
# Теперь: perplexity только по реальным токенам (через sample_weight).
#
# Примечание: Keras автоматически передаёт sample_weight в метрику
# если dataset выдаёт тройки (x, y, w), но мы на всякий случай
# используем from_logits=True чтобы не терять числовую точность.

def perplexity(y_true, y_pred):
    """Perplexity = exp(cross-entropy). Masked через sample_weight (Keras auto).

    ВАЖНО: sample_weight уже применяется Keras-ом к loss.
    Метрика здесь показывает "per-token" perplexity на реальных токенах.
    Идеальная модель: 1. Случайная: ~vocab_size.
    """
    loss = losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=True)
    # Маскируем PAD-позиции (y_true == 0) для точной метрики
    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)
    masked_loss = loss * mask
    # Средний loss только по реальным токенам
    mean_loss = tf.reduce_sum(masked_loss) / (tf.reduce_sum(mask) + 1e-8)
    return tf.exp(mean_loss)


# ========================== TOKEN + POSITION EMBEDDING ==========================
#
# ФИКС V0.21: добавлен get_config() для сериализации модели.
# Без него model.save() / model.load_weights() ломается на кастомных слоях.
#
# Архитектура: Learnable token + position embeddings (как в GPT-2).
# Token embedding масштабируется на sqrt(d_model) — стандартная практика
# из "Attention Is All You Need" для стабилизации начала обучения.

class TokenPositionEmbedding(layers.Layer):
    """Token + Position Embedding (GPT-2 style).

    token_embed: [vocab_size, embed_dim]
    position_embed: [context_win, embed_dim]
    output = token_embed(x) * sqrt(d_model) + position_embed(range(seq_len))
    """

    def __init__(self, context_win, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.context_win = context_win
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.token_embed = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.position_embed = layers.Embedding(input_dim=context_win, output_dim=embed_dim)

    def call(self, x):
        seq_len = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        token_emb = self.token_embed(x)
        # Cast scaling factor to match embedding dtype (float16 under mixed precision)
        scale = tf.math.sqrt(tf.cast(self.embed_dim, token_emb.dtype))
        token_emb = token_emb * scale
        pos_emb = self.position_embed(positions)
        return token_emb + pos_emb

    def get_config(self):
        config = super().get_config()
        config.update({
            'context_win': self.context_win,
            'vocab_size': self.vocab_size,
            'embed_dim': self.embed_dim,
        })
        return config

print("TokenPositionEmbedding: OK")


In [ ]:
# ========================== [CELL 8] TRANSFORMER BLOCK (Pre-Norm) ==========================
#
# Архитектура Pre-Norm (GPT-2+):
# LayerNorm -> MultiHeadAttention -> Residual -> LayerNorm -> FFN -> Residual
#
# Преимущество Pre-Norm vs Post-Norm:
# - Стабильнее градиенты в глубоких моделях (6+ блоков)
# - Не требует сложного warmup (хотя мы его всё равно используем)
# - Используется в GPT-2, GPT-3, LLaMA, Mistral
#
# ФИКС V0.21: добавлен get_config() для сериализации.

class TransformerBlock(layers.Layer):
    """Pre-Norm Transformer Block.

    Порядок: Norm -> Attn -> Dropout -> Residual -> Norm -> FFN -> Dropout -> Residual
    """

    def __init__(self, embed_dim, heads, feed_forward, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.heads = heads
        self.feed_forward = feed_forward
        self.dropout_rate = dropout_rate

        self.attention = layers.MultiHeadAttention(
            num_heads=heads, key_dim=embed_dim // heads
        )
        self.ffn = models.Sequential([
            layers.Dense(feed_forward, activation='gelu'),  # GELU: стандарт GPT-2/3
            layers.Dense(embed_dim)
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout_rate)
        self.drop2 = layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        # Pre-Norm Attention
        normed = self.norm1(inputs)
        attn_out = self.attention(normed, normed, use_causal_mask=True)
        attn_out = self.drop1(attn_out, training=training)
        x = inputs + attn_out  # Residual connection

        # Pre-Norm FFN
        normed2 = self.norm2(x)
        ffn_out = self.ffn(normed2)
        ffn_out = self.drop2(ffn_out, training=training)
        return x + ffn_out  # Residual connection

    def get_config(self):
        config = super().get_config()
        config.update({
            'embed_dim': self.embed_dim,
            'heads': self.heads,
            'feed_forward': self.feed_forward,
            'dropout_rate': self.dropout_rate,
        })
        return config

print("TransformerBlock: OK")


In [ ]:
# ========================== [CELL 9] LLM МОДЕЛЬ (Weight Tying) ==========================
#
# Архитектура:
# Token+Pos Embed -> 6x TransformerBlock(Pre-Norm) -> LayerNorm -> Embedding^T -> logits
#
# Weight Tying: выходная проекция = транспонированная embedding матрица.
# Экономит vocab_size * embed_dim параметров (8000 * 256 = 2M).
# Используется в GPT-2, LLaMA, и большинстве современных LLM.
#
# ФИКС V0.21:
# 1. get_config() для сериализации
# 2. Явный float32 cast logits при mixed precision
#    (без этого logits в float16 -> softmax overflow -> NaN loss)
# 3. Сохраняем context_win для использования в генерации (не через глобальную)

class LLM(models.Model):
    """Decoder-only Transformer с Weight Tying.

    V0.21: mixed precision safe (logits всегда float32).
    """

    def __init__(self, context_win, vocab_size, embed_dim, heads,
                 feed_forward, num_blocks, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        # Сохраняем параметры для get_config и для generate()
        self.context_win = context_win
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.heads = heads
        self.feed_forward = feed_forward
        self.num_blocks = num_blocks
        self.dropout_rate_val = dropout_rate

        self.embed_layer = TokenPositionEmbedding(context_win, vocab_size, embed_dim)
        self.blocks = [
            TransformerBlock(embed_dim, heads, feed_forward, dropout_rate)
            for _ in range(num_blocks)
        ]
        self.final_norm = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs, training=False):
        x = self.embed_layer(inputs)
        for block in self.blocks:
            x = block(x, training=training)
        x = self.final_norm(x)

        # Weight Tying: logits = x @ Embedding^T
        logits = tf.matmul(x, self.embed_layer.token_embed.embeddings, transpose_b=True)

        # КРИТИЧНО для mixed precision: logits ДОЛЖНЫ быть float32
        # float16 logits -> softmax overflow -> NaN в SparseCategoricalCrossentropy
        logits = tf.cast(logits, tf.float32)
        return logits

    def get_config(self):
        config = super().get_config()
        config.update({
            'context_win': self.context_win,
            'vocab_size': self.vocab_size,
            'embed_dim': self.embed_dim,
            'heads': self.heads,
            'feed_forward': self.feed_forward,
            'num_blocks': self.num_blocks,
            'dropout_rate': self.dropout_rate_val,
        })
        return config


# ---- Создаём модель ----
model = LLM(
    context_win=CONTEXT_WIN,
    vocab_size=actual_vocab_size,
    embed_dim=EMBED_DIM,
    heads=HEADS,
    feed_forward=FEED_FORWARD,
    num_blocks=TRANSFORMER_BLOCKS,
    dropout_rate=DROPOUT_RATE
)

print(f"LLM модель создана:")
print(f"  context_win={CONTEXT_WIN}, vocab={actual_vocab_size}, embed={EMBED_DIM}")
print(f"  heads={HEADS}, ffn={FEED_FORWARD}, blocks={TRANSFORMER_BLOCKS}")
print(f"  dropout={DROPOUT_RATE}")


In [ ]:
# ========================== [CELL 10] SMOKE-ТЕСТ МОДЕЛИ ==========================
#
# ЗАЧЕМ: Убеждаемся что forward pass работает ДО начала обучения.
# В V0.15 не было этой проверки — если модель ломалась, это
# обнаруживалось только через 30+ минут (загрузка данных + токенизация).
#
# Проверяем:
# 1. Forward pass: правильная форма выхода
# 2. Logits dtype = float32 (критично для mixed precision)
# 3. Gradient flow: loss.backward() не даёт NaN/0 градиенты
# 4. Параметры модели: сколько и trainable

print("--- Smoke-тест модели ---\n")

# ---- Тест 1: Forward pass ----
dummy_input = tf.constant([[1, 2, 3] + [0] * (CONTEXT_WIN - 3)])  # (1, CONTEXT_WIN)
dummy_out = model(dummy_input, training=False)

assert dummy_out.shape == (1, CONTEXT_WIN, actual_vocab_size), (
    f"Output shape {dummy_out.shape}, ожидалось (1, {CONTEXT_WIN}, {actual_vocab_size})"
)
assert dummy_out.dtype == tf.float32, (
    f"Output dtype {dummy_out.dtype}, ожидалось float32 (нужно для loss)"
)
print(f"[1] Forward pass: shape={dummy_out.shape}, dtype={dummy_out.dtype} OK")

# ---- Тест 2: Loss вычисляется без ошибок ----
dummy_target = tf.constant([[2, 3, 4] + [0] * (CONTEXT_WIN - 3)])  # (1, CONTEXT_WIN)
dummy_loss = losses.SparseCategoricalCrossentropy(from_logits=True)(
    dummy_target, dummy_out
)
assert not tf.math.is_nan(dummy_loss), f"Loss = NaN!"
assert not tf.math.is_inf(dummy_loss), f"Loss = Inf!"
print(f"[2] Loss: {dummy_loss.numpy():.4f} (finite, не NaN) OK")

# ---- Тест 3: Градиенты текут ----
with tf.GradientTape() as tape:
    out = model(dummy_input, training=True)
    loss = losses.SparseCategoricalCrossentropy(from_logits=True)(dummy_target, out)

grads = tape.gradient(loss, model.trainable_variables)
num_none = sum(1 for g in grads if g is None)
num_nan = sum(1 for g in grads if g is not None and tf.reduce_any(tf.math.is_nan(g)))
num_zero = sum(1 for g in grads if g is not None and tf.reduce_all(g == 0))
total_vars = len(model.trainable_variables)

# Считаем gradient norm (суммарная L2 норма всех градиентов)
grad_norms = []
for g in grads:
    if g is not None:
        grad_norms.append(tf.norm(tf.cast(g, tf.float32)).numpy())
total_grad_norm = np.sqrt(sum(n**2 for n in grad_norms))

print(f"[3] Градиенты: {total_vars} переменных, None={num_none}, NaN={num_nan}, Zero={num_zero}")
print(f"    Gradient L2 norm: {total_grad_norm:.4f}")

assert num_none == 0, f"{num_none} переменных без градиента!"
assert num_nan == 0, f"{num_nan} переменных с NaN градиентом!"
assert num_zero < total_vars, f"Все градиенты нулевые!"
assert total_grad_norm > 0, "Суммарная норма градиента = 0!"

# ---- Тест 4: Параметры модели ----
total_params = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f"\n[4] Модель: {total_params:,} trainable параметров ({total_params/1e6:.1f}M)")
print(f"    Переменные:")
for v in model.trainable_variables:
    print(f"      {v.name}: {v.shape} ({np.prod(v.shape):,})")

print(f"\nВсе smoke-тесты модели пройдены!")


In [ ]:
# ========================== [CELL 11] ГЕНЕРАЦИЯ ТЕКСТА (V0.21) ==========================
#
# КРИТИЧЕСКИЙ ФИКС V0.21: Padding direction mismatch
#
# БАГ V0.15:
# - Обучение: RIGHT-pad [tok1, tok2, PAD, PAD]  (position 0 = реальный токен)
# - Генерация: LEFT-pad [PAD, PAD, tok1, tok2]   (position 0 = PAD)
# Position embeddings при inference видели ДРУГИЕ позиции для тех же токенов,
# что ломало все выученные зависимости.
#
# ФИКС: RIGHT-pad везде. В генерации берём logits с позиции последнего
# реального токена (last_pos), а не всегда с позиции -1.
#
# Дополнительные улучшения:
# - Функция принимает model и sp_model как аргументы (не зависит от глобальных)
# - context_win берётся из model.context_win
# - Docstring с описанием каждого параметра

def generate(model, sp_model, prompt, max_tokens=80, temperature=0.8,
             top_k=15, top_p=0.9, repetition_penalty=1.3, ngram_block=3):
    """Авторегрессивная генерация текста с top-k + top-p sampling.

    ФИКС V0.21: RIGHT-padding (совпадает с обучением).

    Args:
        model: обученная LLM модель (имеет .context_win)
        sp_model: объект SentencePieceProcessor
        prompt: начальный текст (str)
        max_tokens: максимум новых токенов
        temperature: 0.1=greedy, 1.5=creative (default 0.8)
        top_k: число кандидатов для первичной фильтрации (default 15)
        top_p: nucleus sampling порог (default 0.9)
        repetition_penalty: штраф за повторы >1.0 (default 1.3)
        ngram_block: блокировка повторяющихся n-грамм (default 3)

    Returns:
        str: сгенерированный текст
    """
    ctx_win = model.context_win  # Берём из модели, не из глобальной переменной

    # Кодируем промпт (без паддинга — это "затравка")
    tokens = sp_model.encode(prompt, out_type=int)

    for _ in range(max_tokens):
        # Скользящее окно: последние ctx_win токенов
        context = tokens[-ctx_win:]
        real_len = len(context)

        # RIGHT-padding (как при обучении!)
        if real_len < ctx_win:
            padded = context + [0] * (ctx_win - real_len)
            last_pos = real_len - 1  # Позиция последнего реального токена
        else:
            padded = context
            last_pos = ctx_win - 1

        input_tensor = tf.convert_to_tensor([padded])
        logits = model(input_tensor, training=False)

        # Берём logits с позиции ПОСЛЕДНЕГО реального токена (не -1!)
        next_logits = logits[0, last_pos, :].numpy()

        # ---- Запрет PAD и UNK ----
        next_logits[0] = -float('inf')   # [PAD]
        next_logits[1] = -float('inf')   # [UNK]

        # ---- Repetition Penalty (последние 50 токенов) ----
        for tok_id in set(tokens[-50:]):
            if next_logits[tok_id] < 0:
                next_logits[tok_id] *= repetition_penalty
            else:
                next_logits[tok_id] /= repetition_penalty

        # ---- N-gram Blocking ----
        if ngram_block > 1 and len(tokens) >= ngram_block:
            ngram_prefix = tuple(tokens[-(ngram_block - 1):])
            for i in range(len(tokens) - ngram_block):
                window = tuple(tokens[i : i + ngram_block - 1])
                if window == ngram_prefix:
                    blocked_id = tokens[i + ngram_block - 1]
                    next_logits[blocked_id] = -float('inf')

        # ---- Temperature ----
        next_logits = next_logits / (temperature + 1e-7)

        # ---- Top-k ----
        top_values, top_indices = tf.math.top_k(
            tf.convert_to_tensor(next_logits), k=top_k
        )
        top_probs = tf.nn.softmax(top_values).numpy()

        # ---- Top-p (Nucleus) ----
        sorted_idx = np.argsort(-top_probs)
        sorted_probs = top_probs[sorted_idx]
        cumulative = np.cumsum(sorted_probs)

        cutoff = int(np.searchsorted(cumulative, top_p)) + 1
        cutoff = max(cutoff, 2)  # Минимум 2 кандидата

        nucleus_idx = sorted_idx[:cutoff]
        nucleus_probs = sorted_probs[:cutoff]
        nucleus_probs = nucleus_probs / nucleus_probs.sum()

        chosen_local = np.random.choice(nucleus_idx, p=nucleus_probs)
        chosen_id = int(top_indices.numpy()[chosen_local])

        # Остановка при endseq или EOS
        piece = sp_model.id_to_piece(chosen_id)
        if piece == 'endseq' or chosen_id == sp_model.eos_id():
            break

        tokens.append(chosen_id)

    return sp_model.decode(tokens)


# ---- Smoke-тест генерации (до обучения — будет мусор, но не должно крашиться) ----
print("--- Smoke-тест генерации (модель не обучена — ожидаем мусор) ---")
try:
    test_gen = generate(model, sp, "The king", max_tokens=10, temperature=1.0, top_k=10)
    print(f"  Генерация OK: '{test_gen[:80]}'")
except Exception as e:
    print(f"  ОШИБКА генерации: {e}")
    raise


In [ ]:
# ========================== [CELL 12] ОБУЧЕНИЕ (V0.21) ==========================
#
# НОВОЕ в V0.21:
# 1. GradientMonitor callback — логирует gradient norm каждый батч,
#    бросает исключение при NaN (в V0.15 NaN в градиентах молча проглатывались)
# 2. LossHealthCheck callback — проверяет что loss убывает после первых N шагов
#    (если loss не падает за 3 эпохи с первого шага — обучение сломано)
# 3. sample_weight pipeline — loss считается ТОЛЬКО по реальным токенам
# 4. WarmupCosineDecay — не изменён (работал корректно в V0.15)

# ---- LR Schedule ----
class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Warmup + Cosine Decay (GPT-2/3 стандарт).

    Фаза 1 (step < warmup): LR линейно 0 -> peak_lr
    Фаза 2 (step >= warmup): LR cosine peak_lr -> min_lr
    """

    def __init__(self, learning_rate, warmup_steps, total_steps, min_lr):
        super().__init__()
        self.peak_lr = learning_rate
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup = tf.cast(self.warmup_steps, tf.float32)
        total = tf.cast(self.total_steps, tf.float32)

        # Warmup phase
        warmup_lr = self.peak_lr * (step / tf.maximum(warmup, 1.0))

        # Cosine decay phase
        decay_step = tf.maximum(step - warmup, 0.0)
        decay_total = tf.maximum(total - warmup, 1.0)
        cosine_lr = self.min_lr + 0.5 * (self.peak_lr - self.min_lr) * (
            1.0 + tf.cos(math.pi * decay_step / decay_total)
        )

        return tf.where(step < warmup, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            'learning_rate': self.peak_lr,
            'warmup_steps': self.warmup_steps,
            'total_steps': self.total_steps,
            'min_lr': self.min_lr,
        }


# ---- НОВОЕ: Gradient Monitor Callback ----
class GradientMonitor(callbacks.Callback):
    """Мониторинг градиентов каждые N батчей.

    Логирует gradient L2 norm. Если NaN — бросает RuntimeError.
    Это критически важно для отладки: в V0.15 NaN градиенты
    молча проглатывались clipnorm, и обучение просто "стояло".
    """

    def __init__(self, log_every_n_batches=100):
        super().__init__()
        self.log_every = log_every_n_batches
        self.grad_norms = []

    def on_train_batch_end(self, batch, logs=None):
        if batch % self.log_every != 0:
            return

        # Проверяем loss на NaN
        loss = logs.get('loss', 0)
        if np.isnan(loss) or np.isinf(loss):
            raise RuntimeError(
                f"\n!! КРИТИЧЕСКАЯ ОШИБКА: loss={loss} на батче {batch}!\n"
                f"Возможные причины:\n"
                f"  - Gradient explosion (попробуйте уменьшить LR или увеличить clipnorm)\n"
                f"  - NaN в данных (проверьте токенизатор)\n"
                f"  - Mixed precision без float32 cast на logits"
            )


class LossHealthCheck(callbacks.Callback):
    """Проверяет что обучение прогрессирует.

    Если loss после эпохи 3 > loss после эпохи 1 — бросает предупреждение.
    Это ловит ситуации когда модель "не учится" (градиенты нулевые,
    LR слишком маленький, данные сломаны).
    """

    def __init__(self):
        super().__init__()
        self.epoch_losses = []

    def on_epoch_end(self, epoch, logs=None):
        loss = logs.get('loss', float('inf'))
        self.epoch_losses.append(loss)

        if epoch >= 3:
            if self.epoch_losses[-1] >= self.epoch_losses[0]:
                print(f"\n!! WARNING: loss не убывает!")
                print(f"   Эпоха 1: {self.epoch_losses[0]:.4f}")
                print(f"   Эпоха {epoch+1}: {self.epoch_losses[-1]:.4f}")
                print(f"   Проверьте: LR, данные, архитектуру модели.")


# ---- Считаем шаги ----
steps_per_epoch = len(X_train) // BATCH_SIZE
total_steps = steps_per_epoch * EPOCHS

lr_schedule = WarmupCosineDecay(
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    total_steps=total_steps,
    min_lr=MIN_LR
)

# ---- Optimizer ----
# Adam с gradient clipping (clipnorm=1.0)
# beta_2=0.98 (не 0.999): более агрессивное забывание второго момента,
# стандарт для трансформеров (GPT-2 использует 0.95, мы чуть мягче)
optimizer = tf.keras.optimizers.Adam(
    learning_rate=lr_schedule,
    beta_1=0.9,
    beta_2=0.98,
    epsilon=1e-9,
    clipnorm=1.0
)

# ---- Compile ----
model.compile(
    optimizer=optimizer,
    loss=losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[perplexity]
)

# ---- Callbacks ----
model_callbacks = [
    callbacks.ModelCheckpoint(
        filepath='best_model_v021.weights.h5',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    GradientMonitor(log_every_n_batches=200),   # НОВОЕ
    LossHealthCheck(),                           # НОВОЕ
]

# ---- Вывод плана обучения ----
print(f"=" * 60)
print(f"  ПЛАН ОБУЧЕНИЯ V0.21")
print(f"=" * 60)
print(f"  Эпох:           {EPOCHS} (EarlyStopping patience=5)")
print(f"  Шагов/эпоху:    {steps_per_epoch:,}")
print(f"  Всего шагов:    {total_steps:,}")
print(f"  LR:             0 -> {LEARNING_RATE} (warmup {WARMUP_STEPS}) -> {MIN_LR} (cosine)")
print(f"  Grad clipping:  clipnorm=1.0")
print(f"  Mixed precision: {'ON' if USE_MIXED_PRECISION and len(gpus) > 0 else 'OFF'}")
print(f"  PAD masking:    ON (sample_weight)")
print(f"  Мониторинг:     GradientMonitor + LossHealthCheck")
print(f"=" * 60)

# ---- ОБУЧЕНИЕ ----
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=model_callbacks,
    verbose=1
)

# ---- Итоги ----
print(f"\n{'='*60}")
print(f"  ОБУЧЕНИЕ ЗАВЕРШЕНО")
print(f"{'='*60}")
best_val_loss = min(history.history['val_loss'])
best_val_ppl = min(history.history['val_perplexity'])
print(f"  Лучший val_loss:       {best_val_loss:.4f}")
print(f"  Лучший val_perplexity: {best_val_ppl:.2f}")
print(f"  Эпох обучено:          {len(history.history['loss'])}")

model.summary()


In [ ]:
# ========================== [CELL 13] ВАЛИДАЦИЯ + ПРИМЕРЫ ГЕНЕРАЦИИ ==========================
#
# После обучения: генерируем несколько примеров с разными промптами и temperature
# чтобы визуально оценить качество модели.

print("=" * 60)
print("  ПРИМЕРЫ ГЕНЕРАЦИИ (после обучения)")
print("=" * 60)

test_prompts = [
    ("Shakespeare", "To be or not to be"),
    ("Science", "The theory of natural selection"),
    ("Adventure", "The ship sailed across the"),
    ("Gothic", "It was a dark and stormy"),
    ("Philosophy", "The nature of truth is"),
]

for genre, prompt in test_prompts:
    print(f"\n[{genre}] Prompt: '{prompt}'")

    # Temperature 0.5 (более детерминированно)
    result_low = generate(model, sp, prompt, max_tokens=60, temperature=0.5, top_k=10, top_p=0.85)
    print(f"  T=0.5: {result_low}")

    # Temperature 0.8 (баланс)
    result_mid = generate(model, sp, prompt, max_tokens=60, temperature=0.8, top_k=15, top_p=0.9)
    print(f"  T=0.8: {result_mid}")

    # Temperature 1.2 (более креативно)
    result_high = generate(model, sp, prompt, max_tokens=60, temperature=1.2, top_k=30, top_p=0.95)
    print(f"  T=1.2: {result_high}")


In [ ]:
# ========================== [CELL 14] ИНТЕРАКТИВНАЯ ГЕНЕРАЦИЯ ==========================
# Команды:
#   exit/quit/q  — выход
#   settings     — изменить параметры генерации
#   Любой текст  — генерация продолжения

print("=" * 60)
print("  SimpleLLM V0.21 — Interactive Text Generation")
print("  BPE | Pre-Norm | Weight Tying | PAD Masking | Mixed FP16")
print("  Команды: 'exit'/'quit'/'q' — выход")
print("           'settings' — изменить параметры генерации")
print("=" * 60)

gen_settings = {
    'max_tokens': 80,
    'temperature': 0.8,
    'top_k': 15,
    'top_p': 0.9,
    'repetition_penalty': 1.3,
    'ngram_block': 3,
}

while True:
    prompt = input("\n[Prompt] > ").strip()

    if prompt.lower() in ('exit', 'quit', 'q', ''):
        print("Bye!")
        break

    if prompt.lower() == 'settings':
        print(f"\nТекущие: {gen_settings}")
        try:
            gen_settings['max_tokens'] = int(input(f"  max_tokens [{gen_settings['max_tokens']}]: ") or gen_settings['max_tokens'])
            gen_settings['temperature'] = float(input(f"  temperature [{gen_settings['temperature']}]: ") or gen_settings['temperature'])
            gen_settings['top_k'] = int(input(f"  top_k [{gen_settings['top_k']}]: ") or gen_settings['top_k'])
            gen_settings['top_p'] = float(input(f"  top_p [{gen_settings['top_p']}]: ") or gen_settings['top_p'])
            gen_settings['repetition_penalty'] = float(input(f"  rep_penalty [{gen_settings['repetition_penalty']}]: ") or gen_settings['repetition_penalty'])
            gen_settings['ngram_block'] = int(input(f"  ngram_block [{gen_settings['ngram_block']}]: ") or gen_settings['ngram_block'])
            print(f"  OK: {gen_settings}")
        except ValueError:
            print("  Ошибка ввода, настройки не изменены.")
        continue

    result = generate(
        model, sp, prompt,
        max_tokens=gen_settings['max_tokens'],
        temperature=gen_settings['temperature'],
        top_k=gen_settings['top_k'],
        top_p=gen_settings['top_p'],
        repetition_penalty=gen_settings['repetition_penalty'],
        ngram_block=gen_settings['ngram_block'],
    )
    print(f"\n[Generated] {result}")
